# Scaled Dot-Product Attention in an Encoder-Decoder Architecture

This notebook walks through the mechanics of **scaled dot-product attention** — the core operation inside every Transformer — and shows how it fits into a naive encoder-decoder design for sequence-to-sequence tasks (e.g. machine translation).

We will:

1. Build intuition for *why* attention is needed.
2. Derive and implement the attention formula step by step.
3. Wire it into a minimal encoder-decoder and train it on a tiny synthetic translation task so you can inspect every intermediate tensor.

## 1 — Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2 — The Attention Formula from First Principles

Given three matrices **Q** (queries), **K** (keys), and **V** (values):

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V
$$

| Symbol | Shape | Meaning |
|--------|-------|---------|
| $Q$    | $(n_q,\; d_k)$ | What the decoder is **looking for** (one row per query position) |
| $K$    | $(n_k,\; d_k)$ | What each encoder position **offers** to be matched against |
| $V$    | $(n_k,\; d_v)$ | The actual **content** that gets aggregated |
| $d_k$  | scalar | Dimension of key/query vectors — used for scaling |

### Why scale by $\sqrt{d_k}$?

When $d_k$ is large the dot products $Q K^\top$ grow in magnitude, pushing the softmax into regions with extremely small gradients. Dividing by $\sqrt{d_k}$ keeps the variance of the logits ≈ 1 regardless of $d_k$.

### 2.1 — Bare-bones implementation

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """Compute scaled dot-product attention.

    Args:
        Q: (batch, n_queries, d_k)
        K: (batch, n_keys,    d_k)
        V: (batch, n_keys,    d_v)

    Returns:
        context:  (batch, n_queries, d_v)  — weighted sum of values
        weights:  (batch, n_queries, n_keys) — attention weights (after softmax)
    """
    d_k = Q.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / (d_k ** 0.5)  # (B, n_q, n_k)
    weights = F.softmax(scores, dim=-1)                        # (B, n_q, n_k)
    context = torch.bmm(weights, V)                            # (B, n_q, d_v)
    return context, weights

### 2.2 — Worked micro-example

Imagine a 4-word source sentence encoded into hidden states $H \in \mathbb{R}^{4 \times 8}$ (sequence length 4, hidden dim 8). The decoder has just produced state $s \in \mathbb{R}^{1 \times 8}$.

We'll use the encoder hidden states as both **K** and **V**, and the decoder state as **Q** (the simplest setup, matching the diagram).

In [ ]:
seq_len, d_model = 4, 8

H = torch.randn(1, seq_len, d_model)   # encoder hidden states
s = torch.randn(1, 1, d_model)          # single decoder state

Q = s   # (1, 1, 8)  — decoder asks: "which encoder positions matter for me?"
K = H   # (1, 4, 8)  — encoder positions present their keys
V = H   # (1, 4, 8)  — and their values

context, weights = scaled_dot_product_attention(Q, K, V)

print("Attention weights (1 query over 4 keys):")
print(weights.squeeze().detach().numpy().round(4))
print(f"\nContext vector shape: {context.shape}  (one vector of dim {d_model})")

### 2.3 — Step-by-step walkthrough

Let's decompose every intermediate tensor so the formula is completely transparent.

In [ ]:
d_k = Q.size(-1)

raw_scores = torch.bmm(Q, K.transpose(1, 2))
print("Step 1 — Raw dot products  QK^T :")
print(f"  Shape: {raw_scores.shape}")
print(f"  Values: {raw_scores.squeeze().detach().numpy().round(4)}")

scaled_scores = raw_scores / (d_k ** 0.5)
print(f"\nStep 2 — Scaled by 1/√d_k  (d_k={d_k}, √d_k={d_k**0.5:.2f}):")
print(f"  Values: {scaled_scores.squeeze().detach().numpy().round(4)}")

attn_weights = F.softmax(scaled_scores, dim=-1)
print(f"\nStep 3 — Softmax → attention weights (sum to 1):")
print(f"  Values: {attn_weights.squeeze().detach().numpy().round(4)}")
print(f"  Sum:    {attn_weights.sum().item():.4f}")

context_vec = torch.bmm(attn_weights, V)
print(f"\nStep 4 — Weighted sum of V → context vector:")
print(f"  Shape: {context_vec.shape}")
print(f"  Values: {context_vec.squeeze().detach().numpy().round(4)}")

### 2.4 — Effect of scaling: why $\sqrt{d_k}$ matters

With large $d_k$ the unscaled dot products blow up and softmax saturates — all the probability mass lands on a single key, starving gradients.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

for ax, d in zip(axes, [8, 64, 512]):
    q = torch.randn(1, 1, d)
    k = torch.randn(1, 10, d)

    raw  = torch.bmm(q, k.transpose(1, 2)).squeeze().detach().numpy()
    scaled = (torch.bmm(q, k.transpose(1, 2)) / d**0.5).squeeze()

    w_raw    = F.softmax(torch.tensor(raw), dim=-1).numpy()
    w_scaled = F.softmax(scaled, dim=-1).squeeze().detach().numpy()

    x = np.arange(10)
    ax.bar(x - 0.15, w_raw, width=0.3, label="unscaled", alpha=0.8)
    ax.bar(x + 0.15, w_scaled, width=0.3, label="scaled", alpha=0.8)
    ax.set_title(f"$d_k = {d}$", fontsize=13)
    ax.set_xlabel("Key index")
    ax.set_ylabel("Attention weight")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.05)

fig.suptitle("Attention weight distributions — unscaled vs. scaled", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

## 3 — Building the Encoder-Decoder with Attention

We now implement the architecture from the diagram:

```
x_9k → x_300 ──▶ Encoder (GRU) ──▶ H  (hidden_dim × seq_len)
                                       │
                      ┌────────────────┘
                      ▼
               Attention(Q=sᵢ, K=H, V=H)
                      │
                      ▼  context c
           Decoder (GRU) ──▶ Y_10k
```

### Dimensions (matching the diagram)

| Tensor | Shape | Notes |
|--------|-------|-------|
| Source embedding | (batch, src_len, 300) | `x_300` |
| Encoder hidden | (batch, src_len, 500) | `h_500`; collected as `H` |
| Decoder hidden | (batch, 1, 500) | `s_500` |
| Attention weights | (batch, 1, src_len) | `w` (one weight per encoder step) |
| Context vector | (batch, 1, 500) | `c_500 = H·w` |
| Target vocab logits | (batch, 10000) | `Y_10k` |

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)               # (B, src_len, embed_dim)
        outputs, hidden = self.rnn(embedded)          # outputs: (B, src_len, hidden_dim)
        return outputs, hidden                        # H, h_last

In [ ]:
class Attention(nn.Module):
    """Scaled dot-product attention between decoder state and encoder outputs.

    Because the decoder hidden state and encoder outputs may live in different
    spaces, we project them into a shared d_k-dimensional space via W_Q and W_K
    (standard practice, even when dimensions happen to match).
    """
    def __init__(self, hidden_dim, d_k=None):
        super().__init__()
        self.d_k = d_k or hidden_dim
        self.W_Q = nn.Linear(hidden_dim, self.d_k, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.d_k, bias=False)

    def forward(self, decoder_state, encoder_outputs):
        """
        Args:
            decoder_state:   (B, 1, hidden_dim)   — current decoder hidden state
            encoder_outputs: (B, src_len, hidden_dim) — all encoder hidden states H

        Returns:
            context: (B, 1, hidden_dim)
            weights: (B, 1, src_len)
        """
        Q = self.W_Q(decoder_state)       # (B, 1, d_k)
        K = self.W_K(encoder_outputs)     # (B, src_len, d_k)
        V = encoder_outputs               # (B, src_len, hidden_dim)

        scores = torch.bmm(Q, K.transpose(1, 2)) / (self.d_k ** 0.5)
        weights = F.softmax(scores, dim=-1)           # (B, 1, src_len)
        context = torch.bmm(weights, V)               # (B, 1, hidden_dim)
        return context, weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = Attention(hidden_dim)
        self.rnn = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward_step(self, token, hidden, encoder_outputs):
        """Decode a single time step."""
        embedded = self.embedding(token)               # (B, 1, embed_dim)

        decoder_state = hidden.transpose(0, 1)         # (B, 1, hidden_dim)
        context, weights = self.attention(decoder_state, encoder_outputs)

        rnn_input = torch.cat([embedded, context], dim=-1)  # (B, 1, embed_dim + hidden_dim)
        output, hidden = self.rnn(rnn_input, hidden)        # output: (B, 1, hidden_dim)

        logits = self.fc_out(output.squeeze(1))        # (B, vocab_size)
        return logits, hidden, weights

In [ ]:
class Seq2SeqWithAttention(nn.Module):
    def __init__(self, encoder, decoder, tgt_sos_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_sos_idx = tgt_sos_idx

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        B, tgt_len = tgt.shape
        tgt_vocab = self.decoder.fc_out.out_features

        encoder_outputs, hidden = self.encoder(src)    # H, h_last

        outputs = torch.zeros(B, tgt_len, tgt_vocab, device=src.device)
        all_weights = []

        token = tgt[:, 0:1]  # <SOS>

        for t in range(tgt_len):
            logits, hidden, weights = self.decoder.forward_step(
                token, hidden, encoder_outputs
            )
            outputs[:, t] = logits
            all_weights.append(weights.detach())

            use_teacher = torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher and t + 1 < tgt_len:
                token = tgt[:, t+1:t+2]
            else:
                token = logits.argmax(dim=-1, keepdim=True)

        all_weights = torch.cat(all_weights, dim=1)    # (B, tgt_len, src_len)
        return outputs, all_weights

## 4 — Toy Translation Task: Number Reversal

To keep things inspectable we define a tiny "translation" problem: **reverse a sequence of digits**.

```
Source: 3 7 1 5     →     Target: 5 1 7 3
```

This is perfect for attention because the model must learn to look at the *last* source token first, then the second-to-last, etc.  We expect the attention heatmap to show a clean anti-diagonal pattern.

In [ ]:
PAD, SOS, EOS = 0, 1, 2
DIGIT_OFFSET = 3  # digits 0-9 are mapped to token ids 3-12
VOCAB_SIZE = 13    # PAD + SOS + EOS + 10 digits


def make_reversal_dataset(n_samples=3000, min_len=4, max_len=8):
    """Generate (source, target) pairs where target = reversed source."""
    sources, targets = [], []
    for _ in range(n_samples):
        length = np.random.randint(min_len, max_len + 1)
        digits = np.random.randint(0, 10, size=length)
        src = [SOS] + (digits + DIGIT_OFFSET).tolist() + [EOS]
        tgt = [SOS] + (digits[::-1] + DIGIT_OFFSET).tolist() + [EOS]
        sources.append(src)
        targets.append(tgt)

    max_src = max(len(s) for s in sources)
    max_tgt = max(len(t) for t in targets)
    for s in sources:
        s.extend([PAD] * (max_src - len(s)))
    for t in targets:
        t.extend([PAD] * (max_tgt - len(t)))

    return torch.tensor(sources), torch.tensor(targets)


def tokens_to_str(ids):
    return " ".join(
        {PAD: "_", SOS: "<s>", EOS: "</s>"}.get(i, str(i - DIGIT_OFFSET))
        for i in ids if i != PAD
    )


src_data, tgt_data = make_reversal_dataset()
print(f"Dataset size: {len(src_data)}")
print(f"Example source: {tokens_to_str(src_data[0].tolist())}")
print(f"Example target: {tokens_to_str(tgt_data[0].tolist())}")

## 5 — Training

In [ ]:
EMBED_DIM  = 32
HIDDEN_DIM = 64
LR         = 3e-3
EPOCHS     = 40
BATCH_SIZE = 128

encoder = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM).to(device)
decoder = Decoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM).to(device)
model   = Seq2SeqWithAttention(encoder, decoder, tgt_sos_idx=SOS).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

n_train = int(0.9 * len(src_data))
train_src, val_src = src_data[:n_train].to(device), src_data[n_train:].to(device)
train_tgt, val_tgt = tgt_data[:n_train].to(device), tgt_data[n_train:].to(device)

history = {"train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    n_batches = 0

    for i in range(0, n_train, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        src_batch = train_src[idx]
        tgt_batch = train_tgt[idx]

        logits, _ = model(src_batch, tgt_batch, teacher_forcing_ratio=0.5)

        loss = criterion(
            logits[:, :-1].reshape(-1, VOCAB_SIZE),
            tgt_batch[:, 1:].reshape(-1)
        )
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    model.eval()
    with torch.no_grad():
        val_logits, _ = model(val_src, val_tgt, teacher_forcing_ratio=0.0)
        val_loss = criterion(
            val_logits[:, :-1].reshape(-1, VOCAB_SIZE),
            val_tgt[:, 1:].reshape(-1)
        ).item()

    history["train_loss"].append(epoch_loss / n_batches)
    history["val_loss"].append(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train loss {epoch_loss/n_batches:.4f} | val loss {val_loss:.4f}")

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training Progress")
plt.legend()
plt.tight_layout()
plt.show()

## 6 — Inspecting Attention Weights

The real payoff: we can **visualize where the decoder looks** at each step.  For the reversal task we expect an anti-diagonal — the decoder should attend to the *last* source digit first and work backwards.

In [ ]:
def greedy_decode(model, src_tensor, max_len=12):
    """Run greedy decoding and return predicted ids + attention weights."""
    model.eval()
    with torch.no_grad():
        enc_out, hidden = model.encoder(src_tensor)
        token = torch.tensor([[SOS]], device=src_tensor.device)
        pred_ids = [SOS]
        all_weights = []

        for _ in range(max_len):
            logits, hidden, weights = model.decoder.forward_step(
                token, hidden, enc_out
            )
            all_weights.append(weights)
            next_id = logits.argmax(dim=-1).item()
            pred_ids.append(next_id)
            if next_id == EOS:
                break
            token = torch.tensor([[next_id]], device=src_tensor.device)

    return pred_ids, torch.cat(all_weights, dim=1).squeeze(0).cpu().numpy()

In [ ]:
def plot_attention(src_tokens, pred_tokens, attn_matrix):
    """Draw an attention heatmap with source on x-axis and prediction on y-axis."""
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.matshow(attn_matrix, cmap="YlOrRd", aspect="auto")

    ax.set_xticks(range(len(src_tokens)))
    ax.set_xticklabels(src_tokens, fontsize=11)
    ax.set_yticks(range(len(pred_tokens)))
    ax.set_yticklabels(pred_tokens, fontsize=11)

    ax.set_xlabel("Source (encoder)", fontsize=12)
    ax.set_ylabel("Predicted (decoder)", fontsize=12)
    ax.xaxis.set_label_position("top")

    for i in range(attn_matrix.shape[0]):
        for j in range(attn_matrix.shape[1]):
            ax.text(j, i, f"{attn_matrix[i, j]:.2f}",
                    ha="center", va="center", fontsize=9,
                    color="white" if attn_matrix[i, j] > 0.5 else "black")

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

In [ ]:
n_examples = 3
indices = np.random.choice(len(val_src), size=n_examples, replace=False)

for idx in indices:
    src_seq = val_src[idx:idx+1]
    pred_ids, attn = greedy_decode(model, src_seq)

    src_labels = [tokens_to_str([t]) for t in src_seq.squeeze().tolist() if t != PAD]
    pred_labels = [tokens_to_str([t]) for t in pred_ids]

    attn_trimmed = attn[:len(pred_labels), :len(src_labels)]

    src_text = tokens_to_str(src_seq.squeeze().tolist())
    pred_text = tokens_to_str(pred_ids)
    tgt_text = tokens_to_str(val_tgt[idx].tolist())

    print(f"Source:    {src_text}")
    print(f"Target:    {tgt_text}")
    print(f"Predicted: {pred_text}")
    print()
    plot_attention(src_labels, pred_labels, attn_trimmed)

## 7 — Accuracy on the Validation Set

In [ ]:
correct = 0
total = len(val_src)

for i in range(total):
    pred_ids, _ = greedy_decode(model, val_src[i:i+1])
    target_ids = [t for t in val_tgt[i].tolist() if t != PAD]
    if pred_ids == target_ids:
        correct += 1

print(f"Exact-match accuracy: {correct}/{total} = {correct/total:.1%}")

## 8 — Inside the Attention Head: Manual Computation

To solidify understanding, let's manually walk through what the `Attention` module does on a single example — extracting the learned $W_Q$ and $W_K$ matrices and replicating every step outside the module.

In [ ]:
sample_src = val_src[0:1]
model.eval()

with torch.no_grad():
    enc_out, h = model.encoder(sample_src)  # enc_out: (1, src_len, 64)

print(f"Encoder outputs H — shape: {enc_out.shape}")
print(f"Last encoder hidden — shape: {h.shape}")

s = h.transpose(0, 1)  # (1, 1, 64) — initial decoder state used as query

W_Q = model.decoder.attention.W_Q.weight  # (d_k, hidden_dim)
W_K = model.decoder.attention.W_K.weight
d_k = W_Q.shape[0]

Q = s @ W_Q.T           # (1, 1, d_k)
K = enc_out @ W_K.T     # (1, src_len, d_k)
V = enc_out              # (1, src_len, hidden_dim)

print(f"\nQ shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

scores = torch.bmm(Q, K.transpose(1, 2)) / (d_k ** 0.5)
print(f"\nScaled scores QK^T/√d_k: {scores.squeeze().numpy().round(4)}")

weights = F.softmax(scores, dim=-1)
print(f"Attention weights:        {weights.squeeze().numpy().round(4)}")

context = torch.bmm(weights, V)
print(f"\nContext vector c — shape: {context.shape}")
print(f"Context (first 8 dims):   {context.squeeze()[:8].numpy().round(4)}")

## 9 — Key Takeaways

1. **Scaled dot-product attention** computes a soft lookup table: the query asks "which keys match me?", and the answer is a weighted combination of values.

2. The $\frac{1}{\sqrt{d_k}}$ scaling is critical for stable training — without it, softmax saturates in high dimensions.

3. In an **encoder-decoder** model the query comes from the decoder state and the keys/values come from encoder hidden states.  The resulting context vector lets the decoder focus on different parts of the input at every output step.

4. The attention heatmap for the reversal task shows a clean **anti-diagonal** — empirical proof that the mechanism learns to look at the right source position for each target position.

5. This single-head attention is the building block for **multi-head attention** in Transformers, where several heads run in parallel and their outputs are concatenated.